
# Previsão de Expansão Urbana com Random Forest

- Pré-processamento de imagens VIIRS
- Extração de características espaciais
- Treinamento de modelo Random Forest
- Validação temporal
- Geração de mapa futuro de crescimento urbano

---



# Imports


In [1]:

import cv2
import numpy as np
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, jaccard_score



# Constantes


In [2]:
BASE_PATH = os.getcwd()

# Janela local
KSIZE = 7

# Densidade de amostragem
STEP = 2

# Limiar base de crescimento
THRESHOLD_BASE = 5.0

# Horizonte futuro da previsão
FUTURE_DELTA_YEARS = 10



# Funções utilitárias


In [ ]:

def baixar_imagem(filepath):
    img = cv2.imread(filepath, cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(f"Erro ao carregar imagem: {filepath}")

    img = np.nan_to_num(img, nan=0.0)

    return img


def salvar_imagem(path, image):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, image)



# Pré-processamento das imagens


In [ ]:
def preprocessar_img(input_data, output_filename, threshold_value=15):

    if isinstance(input_data, str):
        img = cv2.imread(input_data, cv2.IMREAD_UNCHANGED)

        if img is None:
            raise FileNotFoundError(input_data)
    else:
        img = input_data.copy()

    img = np.nan_to_num(img, nan=0.0)

    img[img < 0] = 0

    max_val = np.percentile(img, 99.9)
    img = np.clip(img, 0, max_val)

    img_normalizada = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    _, img_binarizada = cv2.threshold(img_normalizada, threshold_value, 255, cv2.THRESH_BINARY)

    output_dir = os.path.join(BASE_PATH, "generated_images")

    path_normalizada = os.path.join(output_dir, f"normalizada_{output_filename}")
    path_binarizada = os.path.join(output_dir, f"binaria_{output_filename}")

    salvar_imagem(path_normalizada, img_normalizada)
    salvar_imagem(path_binarizada, img_binarizada)

    return img_binarizada, img_normalizada


# Carregamento anual das imagens VIIRS


In [ ]:
def carregar_imagem(regiao, ano, mes):
    
    caminho_do_arquivo = os.path.join(
        BASE_PATH,
        "data",
        "NTL_LITORAL_SC",
        "QGIS_LITORAL",
        "RASTER",
        "NTL_LITORAL",
        f"NTL_{ano}",
        f"VIIRS_NTL_MedianaMensal_{regiao}_{ano}_{mes}_reprojetada.tif",
    )

    if os.path.exists(caminho_do_arquivo):

        img = cv2.imread(caminho_do_arquivo, cv2.IMREAD_UNCHANGED)

        if img is None:
            return None

        return preprocessar_img(img, f"{regiao}_{ano}_{mes}.png")


# Extração de características


In [ ]:

def build_training_data(available_images, img_dem):

    years_list = sorted(available_images.keys())

    X_train = []
    y_train = []

    dem_float = img_dem.astype(np.float32)

    mean_dem = cv2.blur(dem_float, (KSIZE, KSIZE))

    mean_sq_dem = cv2.blur(dem_float**2, (KSIZE, KSIZE))

    var_dem = np.maximum(mean_sq_dem - mean_dem**2, 0)

    sobelx = cv2.Sobel(dem_float, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(dem_float, cv2.CV_32F, 0, 1, ksize=3)

    slope_dem = cv2.magnitude(sobelx, sobely)

    slope_blur = cv2.blur(slope_dem, (KSIZE, KSIZE))

    for idx in range(len(years_list) - 1):

        y1 = years_list[idx]
        y2 = years_list[idx + 1]

        delta_years = y2 - y1

        img1 = available_images[y1].astype(np.float32)
        img2 = available_images[y2].astype(np.float32)

        threshold = (THRESHOLD_BASE / 10.0) * delta_years

        mean1 = cv2.blur(img1, (KSIZE, KSIZE))

        mean_sq1 = cv2.blur(img1**2, (KSIZE, KSIZE))

        var1 = np.maximum(mean_sq1 - mean1**2, 0)

        macro1 = cv2.blur(img1, (11, 11))

        mean2 = cv2.blur(img2, (KSIZE, KSIZE))

        diff = mean2 - mean1

        m1 = mean1[::STEP, ::STEP].flatten()
        v1 = var1[::STEP, ::STEP].flatten()

        macro = macro1[::STEP, ::STEP].flatten()

        md = mean_dem[::STEP, ::STEP].flatten()
        vd = var_dem[::STEP, ::STEP].flatten()
        sd = slope_blur[::STEP, ::STEP].flatten()

        local_diff = m1 - macro

        df = diff[::STEP, ::STEP].flatten()

        delta = np.full_like(m1, delta_years)

        features = np.column_stack((
            m1,
            v1,
            macro,
            local_diff,
            md,
            vd,
            sd,
            delta
        ))

        labels = (df > threshold).astype(int)

        X_train.extend(features.tolist())
        y_train.extend(labels.tolist())

    return np.array(X_train), np.array(y_train)



# Treinamento do modelo


In [8]:

def train_model(X_train, y_train):

    model = RandomForestClassifier(
        n_estimators=500,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    return model



# Validação temporal


In [9]:

def validate_model(model, available_images):

    if 2020 not in available_images or 2024 not in available_images:
        print("Anos necessários para validação não encontrados.")
        return

    img2020 = available_images[2020].astype(np.float32)
    img2024 = available_images[2024].astype(np.float32)

    mean2020 = cv2.blur(img2020, (KSIZE, KSIZE))

    diff = cv2.blur(img2024, (KSIZE, KSIZE)) - mean2020

    y_true = (diff > 2).astype(int).flatten()

    X_val = np.column_stack((
        mean2020.flatten(),
        np.zeros_like(mean2020.flatten()),
        mean2020.flatten(),
        np.zeros_like(mean2020.flatten()),
        np.zeros_like(mean2020.flatten()),
        np.zeros_like(mean2020.flatten()),
        np.zeros_like(mean2020.flatten()),
        np.full_like(mean2020.flatten(), 4)
    ))

    y_pred = model.predict(X_val)

    f1 = f1_score(y_true, y_pred, zero_division=0)

    iou = jaccard_score(y_true, y_pred)

    print(f"F1 Score: {f1:.4f}")
    print(f"IoU: {iou:.4f}")



# Geração do mapa futuro


In [ ]:

def generate_future_map(model, available_images, img_dem):

    years = sorted(available_images.keys())

    last_year = years[-1]

    img_last = available_images[last_year].astype(np.float32)

    mean_last = cv2.blur(img_last, (KSIZE, KSIZE))

    var_last = np.maximum(
        cv2.blur(img_last**2, (KSIZE, KSIZE)) - mean_last**2,
        0
    )

    macro_last = cv2.blur(img_last, (11, 11))

    features = np.column_stack((
        mean_last.flatten(),
        var_last.flatten(),
        macro_last.flatten(),
        (mean_last.flatten() - macro_last.flatten()),
        img_dem.flatten(),
        np.zeros_like(img_dem.flatten()),
        np.zeros_like(img_dem.flatten()),
        np.full_like(mean_last.flatten(), FUTURE_DELTA_YEARS)
    ))

    preds = model.predict(features)

    growth_map = preds.reshape(img_last.shape).astype(np.uint8) * 255

    kernel = np.ones((3, 3), np.uint8)

    growth_map = cv2.morphologyEx(
        growth_map,
        cv2.MORPH_OPEN,
        kernel
    )

    growth_map = cv2.morphologyEx(
        growth_map,
        cv2.MORPH_CLOSE,
        kernel
    )

    output_dir = os.path.join(BASE_PATH, "generated_images")

    output_path = os.path.join(
        output_dir,
        "mapa_previsao_futura.png"
    )

    salvar_imagem(output_path, growth_map)

    print(f"Mapa salvo em: {output_path}")

    return growth_map



# Execução do pipeline completo


In [11]:

available_images = load_yearly_images()

years = sorted(available_images.keys())

print("Anos carregados:", years)


Processando 2015...
Processando 2016...
Processando 2017...
Processando 2018...
Processando 2019...
Processando 2020...
Processando 2021...
Processando 2022...
Processando 2023...
Processando 2024...
Processando 2025...
Anos carregados: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [12]:

img_dem = load_dem(
    list(available_images.values())[0]
)


DEM não encontrado. Utilizando DEM zerado.


In [13]:

X_train, y_train = build_training_data(
    available_images,
    img_dem
)

print("Total de amostras:", len(X_train))


ValueError: operands could not be broadcast together with shapes (178,101) (185,101) 

In [ ]:

model = train_model(
    X_train,
    y_train
)

print("Modelo treinado.")


In [ ]:

validate_model(
    model,
    available_images
)


In [ ]:

future_map = generate_future_map(
    model,
    available_images,
    img_dem
)
